<a href="https://colab.research.google.com/github/RafihaikalP/Rafi-Haikal-Pratama_2411532002_ML_2526/blob/main/Praktikum5/TugasCrossValidation_ipnyb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [6]:
train_url = 'https://raw.githubusercontent.com/RafihaikalP/Rafi-Haikal-Pratama_2411532002_ML_2526/refs/heads/main/Praktikum5/train.csv'

test_url = 'https://raw.githubusercontent.com/RafihaikalP/Rafi-Haikal-Pratama_2411532002_ML_2526/refs/heads/main/Praktikum5/test.csv'

submission_url = 'https://raw.githubusercontent.com/RafihaikalP/Rafi-Haikal-Pratama_2411532002_ML_2526/refs/heads/main/Praktikum5/gender_submission.csv'

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)
submission_df = pd.read_csv(submission_url)

print("Ukuran train.csv:", train_df.shape)
print("Ukuran test.csv:", test_df.shape)
print("Ukuran gender_submission.csv:", submission_df.shape)

Ukuran train.csv: (891, 12)
Ukuran test.csv: (418, 11)
Ukuran gender_submission.csv: (418, 2)


In [7]:
train_df.head()
test_df.head()
submission_df.head()
train_df.info()
train_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [8]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = train_df[features]
y = train_df["Survived"]

X_test_final = test_df[features]

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)
print("Ukuran X_test_final:", X_test_final.shape)

Ukuran X: (891, 7)
Ukuran y: (891,)
Ukuran X_test_final: (418, 7)


In [9]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

In [11]:
k_values = [5, 10]
scoring = ["accuracy", "precision", "recall", "f1"]

all_results = {}

for k in k_values:
    print(f"===== HASIL K-FOLD CROSS VALIDATION K = {k} =====")

    kf = KFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    cv_results = cross_validate(
        model,
        X,
        y,
        cv=kf,
        scoring=scoring
    )

    all_results[k] = cv_results

    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        print(f"{metric.capitalize()} tiap fold:", scores)
        print(f"Rata-rata {metric}:", scores.mean())
        print(f"Standar deviasi {metric}:", scores.std())
        print()

===== HASIL K-FOLD CROSS VALIDATION K = 5 =====
Accuracy tiap fold: [0.81005587 0.79775281 0.84269663 0.7752809  0.78089888]
Rata-rata accuracy: 0.8013370158809867
Standar deviasi accuracy: 0.024067078255205286

Precision tiap fold: [0.78571429 0.79245283 0.80882353 0.70967742 0.6969697 ]
Rata-rata precision: 0.7587275523278532
Standar deviasi precision: 0.04603310397694564

Recall tiap fold: [0.74324324 0.62686567 0.78571429 0.66666667 0.70769231]
Rata-rata recall: 0.7060364349916589
Standar deviasi recall: 0.05577389500081997

F1 tiap fold: [0.76388889 0.7        0.79710145 0.6875     0.70229008]
Rata-rata f1: 0.7301560829000258
Standar deviasi f1: 0.04272008741819369

===== HASIL K-FOLD CROSS VALIDATION K = 10 =====
Accuracy tiap fold: [0.84444444 0.7752809  0.82022472 0.76404494 0.79775281 0.84269663
 0.78651685 0.74157303 0.73033708 0.84269663]
Rata-rata accuracy: 0.7945568039950064
Standar deviasi accuracy: 0.04011402060507459

Precision tiap fold: [0.78947368 0.78125    0.838709

In [12]:
comparison_rows = []

for k in k_values:
    cv_results = all_results[k]

    row = {
        "K": k,
        "Accuracy Mean": cv_results["test_accuracy"].mean(),
        "Accuracy Std": cv_results["test_accuracy"].std(),
        "Precision Mean": cv_results["test_precision"].mean(),
        "Precision Std": cv_results["test_precision"].std(),
        "Recall Mean": cv_results["test_recall"].mean(),
        "Recall Std": cv_results["test_recall"].std(),
        "F1 Mean": cv_results["test_f1"].mean(),
        "F1 Std": cv_results["test_f1"].std()
    }

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,K,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std
0,5,0.801337,0.024067,0.758728,0.046033,0.706036,0.055774,0.730156,0.042720
1,10,0.794557,0.040114,0.748655,0.054776,0.697322,0.098128,0.718579,0.063877


In [13]:
for k in k_values:
    print(f"===== CONFUSION MATRIX DAN CLASSIFICATION REPORT K = {k} =====")

    kf = KFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    y_pred_cv = cross_val_predict(
        model,
        X,
        y,
        cv=kf
    )

    cm = confusion_matrix(y, y_pred_cv)
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y, y_pred_cv))
    print()

===== CONFUSION MATRIX DAN CLASSIFICATION REPORT K = 5 =====
Confusion Matrix:
[[472  77]
 [100 242]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       549
           1       0.76      0.71      0.73       342

    accuracy                           0.80       891
   macro avg       0.79      0.78      0.79       891
weighted avg       0.80      0.80      0.80       891


===== CONFUSION MATRIX DAN CLASSIFICATION REPORT K = 10 =====
Confusion Matrix:
[[470  79]
 [104 238]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84       549
           1       0.75      0.70      0.72       342

    accuracy                           0.79       891
   macro avg       0.78      0.78      0.78       891
weighted avg       0.79      0.79      0.79       891




In [16]:
test_predictions = final_model.predict(X_test_final)

test_predictions[:10]

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [17]:
final_submission = submission_df.copy()

final_submission["Survived"] = test_predictions

final_submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
